# Train the Yeo phone recognizer on CMU ARCTIC (Colab GPU)



Self-contained: embeds the **bug-fixed** repo code, downloads public CMU ARCTIC,

force-aligns it with **MFA** to get IPA phone segments, trains the **conv-only**

XLS-R recognizer head on GPU, and exports a checkpoint already patched to load on

your local CPU/Windows machine. **No patient data is used** — training is on public

clean speech only.



### Before you run

1. **Runtime -> Change runtime type -> T4 GPU**, then run cells top to bottom.

2. Cell **#3 (condacolab) RESTARTS the kernel.** That is expected. After it restarts,

   just continue from cell **#4** onward (do not re-run 1-3).

3. The whole run is roughly **30-60 min**, most of it MFA alignment + training.

## 1. Check the GPU

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
!nvidia-smi -L || echo 'No GPU -> set Runtime > Change runtime type > T4 GPU'

ERROR: Error in parse(text = input): <text>:1:8: unexpected symbol
1: import torch
           ^


## 2. Config (edit here)

`SPEAKERS`: CMU ARCTIC voices to train on. `bdl`(m)+`slt`(f) is a gender-balanced

pair; add `clb`(f)+`rms`(m) for the 4-speaker 'strong'-style set.

In [ ]:
SPEAKERS   = ['bdl', 'slt']       # add 'clb','rms' for the 4-speaker set
NUM_EPOCHS = 15
BATCH_SIZE = 8
LR         = 1e-3                 # README uses 1e-3 with phonewise_average
EXP_NAME   = 'arctic_conv_colab'
import os; os.makedirs('/content/dysarthria-gop', exist_ok=True)

## 3. Install MFA (via condacolab) — **THIS RESTARTS THE KERNEL**

After the restart, skip straight to cell #4.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()   # <-- kernel restarts here; then continue at cell #4

## 4. (after restart) Dependencies: MFA + Python libs

In [ ]:
# conda env now active. Install MFA and the Python deps the training code needs.
!conda install -y -c conda-forge montreal-forced-aligner 2>&1 | tail -3
!pip install -q torch transformers librosa soundfile praatio textgrids tensorboard scikit-learn pandas numpy tqdm
import torch, transformers, librosa, praatio
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| transformers', transformers.__version__)

## 5. Write the bug-fixed project code

These are the corrected files (A1 log_softmax loss, A5 collator length, A7 str2bool /

full-transformer patch). `num_workers` is lowered to 2 for Colab.

In [ ]:
%%writefile /content/dysarthria-gop/dataset.py
import argparse
from itertools import product
from pathlib import Path

import librosa
import pandas as pd
import praatio.textgrid
import textgrids
from tqdm import tqdm

import torch


def _get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset_path", type=Path, help="Path to dataset.")
    parser.add_argument("--dataset_type", type=str, choices=["commonphone", "ssnce", "torgo", "qolt", "l2arctic", "uaspeech"])
    parser.add_argument("--output_path", type=Path, help="Output csv folder")
    return parser.parse_args()


def get_vocab(df):
    vocabs = df["phone"].unique()
    index_to_vocab = dict(enumerate(sorted(vocabs)))
    vocab_to_index = {v: i for i, v in index_to_vocab.items()}
    return index_to_vocab, vocab_to_index


def reduce_vocab(df):
    reducer = {'(...)': '(...)', 'a': 'a', 'aɪ': 'aɪ', 'aʊ': 'aʊ', 'aː': 'a', 'ã': 'a', 'ɐ': 'a', 'ɑ': 'a', 'ɑː': 'a', 'ɒ': 'a', 'ɔ': 'ɔ', 'ɔː': 'ɔ', 'ʀ': 'ʀ', 'ʊ': 'ʊ', 'ʊə': 'ʊə', 'θ': 'θ', 'ʃ': 'ʃ', 'ʃʲ': 'ʃ', 'ʃː': 'ʃ', 'ɨ': 'ɨ', 'ɔɪ': 'ɔɪ', 'ɔʏ': 'ɔɪ', 'ə': 'ʌ', 'ɜː': 'ʌ', 'ʌ': 'ʌ', 'əʊ': 'əʊ', 'ɪə': 'ɪə', 'ɥ': 'ɥ', 'ɛ': 'ɛ', 'ɛː': 'ɛ', 'b': 'b', 'bʲ': 'b', 'bː': 'b', 'd': 'd', 'dʲ': 'd', 'dː': 'd', 'ɡ': 'ɡ', 'ɡʲ': 'g', 'ɡː': 'g', 'dz': 'dʒ', 'dːz': 'dz', 'ɣ': 'ɣ', 'dʒ': 'dʒ', 'dːʒ': 'dʒ', 'ʒ': 'dʒ', 'e': 'e', 'eː': 'e', 'ẽ': 'e', 'ei': 'eɪ', 'eə': 'eə', 'eɪ': 'eɪ', 'f': 'f', 'fʲ': 'f', 'fː': 'f', 'h': 'h', 'i': 'i', 'iː': 'i', 'ɪ': 'i', 'y': 'i', 'yː': 'i', 'ʏ': 'i', 'j': 'j', 'k': 'k', 'kʲ': 'k', 'kː': 'k', 'l': 'l', 'lʲ': 'l', 'lː': 'l', 'm': 'm', 'mʲ': 'm', 'mː': 'm', 'ɱ': 'm', 'n': 'n', 'nʲ': 'n', 'nː': 'n', 'ɲ': 'n', 'ɲː': 'n', 'o': 'o', 'oː': 'o', 'õ': 'œ', 'p': 'p', 'pʲ': 'p', 'pː': 'p', 'r': 'r', 'rʲ': 'r', 'rː': 'r', 's': 's', 'sʲ': 's', 'sː': 's', 'ç': 's', 't': 't', 'tʲ': 't', 'tː': 't', 'ts': 'ts', 'tːs': 'ts', 'tʃ': 'tʃ', 'tʃʲ': 'tʃ', 'tːʃ': 'tʃ', 'u': 'u', 'uː': 'u', 'v': 'v', 'vʲ': 'v', 'vː': 'v', 'β': 'v', 'w': 'w', 'x': 'x', 'xʲ': 'x', 'z': 'z', 'zʲ': 'z', 'æ': 'æ', 'ð': 'ð', 'ŋ': 'ŋ', 'œ': 'œ', 'ø': 'œ', 'øː': 'œ', 'ɶ̃': 'œ', 'ʎ': 'ʎ', 'ʎː': 'ʎ', 'ʔ': 'ʔ', 'ʝ': 'ɣ'}
    df["phone"] = df["phone"].apply(lambda p: reducer[p])
    return df


def _prepare_commonphone(commonphone_path: Path):
    langs = ("de", "en", "es", "fr", "it", "ru")
    splits = ("train", "dev", "test")
    rows = []
    for lang, split in tqdm(product(langs, splits)):
        cp_path = commonphone_path / lang
        df = pd.read_csv(cp_path / f"{split}.csv")

        for _, row in df.iterrows():
            audio_file_name = cp_path / "wav" / row["audio file"].replace(".mp3", ".wav")
            grid_file_name = cp_path / "grids" / f"{audio_file_name.stem}.TextGrid"

            grid = textgrids.TextGrid(grid_file_name)
            for g in grid["MAU"]:
                rows.append({
                    "audio": audio_file_name,
                    "language": lang,
                    "id": row["id"],
                    "split": split,
                    "sentence": row["text"],
                    "min": g.xmin,
                    "max": g.xmax,
                    "phone": g.text,
                })
    return pd.DataFrame(rows)


def _prepare_ssnce(ssnce_path: Path):
    # labels = ("0", "1", "2", "3", "4", "5", "6")
    labels = ("0_healthy", "1_mild", "2_moderate", "3_severe")
    vocab = {'#': '(...)', 'SIL': '(...)', 'a': 'a', 'aa': 'aː', 'ai': 'aɪ', 'b': 'b', 'c': 'tʃ', 'dx': 'd', 'd': 'ð', 'e': 'e', 'ee': 'eː', 'g': 'ɡ', 'h': 'h', 'i': 'i', 'ii': 'iː', 'j': 'dʒ', 'k': 'k', 'l': 'l', 'lx': 'l', 'm': 'm', 'n': 'n', 'nx': 'n', 'nd': 'n', 'nj': 'ɲ', 'ng': 'ŋ', 'o': 'o', 'oo': 'oː', 'p': 'p', 'r': 'r', 'rx': 'r', 's': 's', 'sx': 'ʃ', 'tx': 't', 't': 'θ', 'u': 'u', 'uu': 'uː', 'y': 'ʝ', 'eu': 'ɨ', 'zh': 'r', 'w': 'w'}
    rows = []
    for label in labels:
        index, label_name = label.split("_")
        index = int(index)

        for audio_path in (ssnce_path / label).glob("*.wav"):
            grid_path = audio_path.with_suffix(".TextGrid")
            grid = praatio.textgrid.openTextgrid(grid_path, includeEmptyIntervals=False)
            audio_len = librosa.get_duration(filename=audio_path, sr=16000)
            for entry in grid.getTier("phonemes").entries:
                if entry.end <= audio_len:
                    rows.append({
                        "audio": audio_path,
                        "label": index,
                        "label_name": label_name,
                        "min": entry.start,
                        "max": entry.end,
                        "phone": vocab[entry.label],
                    })
                else:
                    break
            rows[-1]["max"] = audio_len
    return pd.DataFrame(rows)


def _prepare_torgo(torgo_path: Path):
    labels = ("0_healthy", "1_mild", "2_moderate", "3_severe")
    vocab = {'': '(...)', '#': '(...)', '@': '(...)', 'OY1': 'ɔɪ', 'T': 't', 'UH1': 'ʊ', 'ZH': 'ʒ', 'A01': 'ɒ', 'SH': 'ʃ', 'G': 'ɡ', 'B': 'b', 'k': 'k', 'AA2': 'ɑ', 'EY1': 'eɪ', 'AA': 'ɑ', 'EI': 'eɪ', 'AW1': 'aʊ', 'ER0': 'eə', 'Y': 'y', 'IH2': 'ɪ', 'AE2': 'æ', 'CH': 'tʃ', 'A02': 'ɒ', 'AE': 'æ', 'UW2': 'u', 'DH': 'ð', 'EH0': 'ɛ', 'P': 'p', 'F': 'f', 'AH0': 'ə', 'E': '(...)', 'd': 'd', 'EY2': 'eɪ', 'OW0': 'əʊ', 'IY0': 'i', 'NG': 'ŋ', 'AE0': 'æ', 'JH': 'dʒ', 'AY1': 'aɪ', 'AO1': 'ɒ', 'OW2': 'əʊ', 'EY': 'æ', 'sp': '(...)', 'sil': '(...)', 'IY1': 'i', 'AH2': 'ɐ', 'ER1': 'eə', 'OW1': 'əʊ', 'K': 'k', 'L': 'l', 'EH1': 'ɛ', 'OU': 'əʊ', 'AA1': 'ɑ', 'AY': 'aɪ', 'AW2': 'aʊ', 'R': 'r', 'HH': 'h', 'H': 'h', 'IH1': 'ɪ', 'UW1': 'u', 'S': 's', '3': '(...)', 'EY0': 'eɪ', 'IH0': 'ɪ', 'Z': 'z', 'W': 'w', 'AH1': 'ɐ', 'AY2': 'aɪ', 'N': 'n', 'AO2': 'ɒ', 'UW': 'u', 'EH2': 'ɛ', 'V': 'v', 'U': 'u', 'IY2': 'i', 'A0': 'ɒ', 'AE1': 'æ', 'UW0': 'u', 'TH': 'θ', 'M': 'm', 'A': 'ɑ', 'D': 'd', 'UH': 'ɐ', 'IH': 'ɪ', 'ER': 'eə', 'AH': '(...)', 'OW': 'aʊ', 'IY': 'i', 'EH': 'ɛ', 'AI': 'aɪ', 'AW': 'aʊ'}
    diphthongs = {'OY1', 'EY1', 'EI', 'AW1', 'EY2', 'OW0', 'AY1', 'OW2', 'OW1', 'OU', 'AY', 'AW2', 'EY0', 'AY2', 'OW', 'AI', 'AW', 'ER', 'ER0', 'ER1'}
    rows = []
    for label in labels:
        index, label_name = label.split("_")
        index = int(index)

        for audio_path in (torgo_path / label).glob("*.wav"):
            grid_path = audio_path.with_suffix(".TextGrid")
            grid = praatio.textgrid.openTextgrid(grid_path, includeEmptyIntervals=False)
            for entry in grid.getTier("phones").entries:
                if entry.label not in diphthongs:
                    rows.append({
                        "audio": audio_path,
                        "label": index,
                        "label_name": label_name,
                        "min": entry.start,
                        "max": entry.end,
                        "phone": vocab[entry.label],
                    })
                else:
                    rows.append({
                        "audio": audio_path,
                        "label": index,
                        "label_name": label_name,
                        "min": entry.start,
                        "max": (entry.start + entry.end) / 2,
                        "phone": vocab[entry.label][0],
                    })
                    rows.append({
                        "audio": audio_path,
                        "label": index,
                        "label_name": label_name,
                        "min": (entry.start + entry.end) / 2,
                        "max": entry.end,
                        "phone": vocab[entry.label][1],
                    })

            audio_length = librosa.get_duration(filename=audio_path, sr=16000)
            if entry.end < audio_length:
                rows.append({
                    "audio": audio_path,
                    "label": index,
                    "label_name": label_name,
                    "min": entry.end,
                    "max": audio_length,
                    "phone": "(...)",
                })
    return pd.DataFrame(rows)


def _prepare_uaspeech(uaspeech_path: Path):
    labels = ("0_healthy", "1_mild", "2_moderate", "3_severe", "4_profound")
    vocab  = {
        "spn": "(...)", "aj": "aɪ", "aw": "aʊ",
        "c": "k", "cʰ": "k", "d̪": "d", "ej": "(...)",
        "kʰ": "k", "m̩": "m", "n̩": "n", "ow": "əʊ",
        "pʰ": "p", "tʰ": "t", "t̪": "t", "ɒː": "ɒ",
        "ɔj": "ɔɪ", "əw": "əʊ", "ɚ": "r", "ɝ": "r", 
        "ɟ": "k", "ɫ": "l", "ɫ̩": "l", "ɹ": "r", "ɾ": "r",
        "ʉ": "u", "ʉː": "u", 
    }

    rows = []
    for label in labels:
        index, label_name = label.split("_")
        index = int(index)

        for audio_path in (uaspeech_path / label).glob("**/*.wav"):
            grid_path = audio_path.with_suffix(".TextGrid")
            if not grid_path.exists():
                print(grid_path)
                continue
            grid = praatio.textgrid.openTextgrid(grid_path, includeEmptyIntervals=False)
            for entry in grid.getTier("phones").entries:
                rows.append({
                    "audio": audio_path,
                    "label": index,
                    "label_name": label_name,
                    "min": entry.start,
                    "max": entry.end,
                    "phone": vocab.get(entry.label, entry.label),
                })
            audio_length = librosa.get_duration(filename=audio_path, sr=16000)
            if entry.end < audio_length:
                rows.append({
                    "audio": audio_path,
                    "label": index,
                    "label_name": label_name,
                    "min": entry.end,
                    "max": audio_length,
                    "phone": "(...)",
                })
    return pd.DataFrame(rows)


def _prepare_qolt(qolt_path: Path):
    labels = ("0_healthy", "1_mild", "2_mildmod", "3_modsev", "4_sev")
    vocab = {'#': '(...)', '': '(...)', 'd': 'd', 'dʑ': 'ts', 'e': 'e', 'h': 'h', 'i': 'i', 'j': 'j', 'k': 'k', 'k̚': 'k', 'k͈': 'k', 'm': 'm', 'n': 'n', 'o': 'o', 'oː': 'o', 'p': 'p', 'pʰ': 'p', 'p̚': 'p', 'p͈': 'p', 'spn': '(...)', 'sʰ': 's', 's͈': 's', 't': 't', 'tɕ': 'ts', 'tɕʰ': 'ts', 'tʰ': 't', 't̚': 't', 'u': 'u', 'w': 'w', 'ŋ': 'ŋ', 'ɐ': 'ɐ', 'ɕʰ': 's', 'ɛː': 'ɛ', 'ɡ': 'ɡ', 'ɨ': 'ɨ', 'ɭ': 'l', 'ɸ': 'h', 'ɾ': 'r', 'ʌ': 'ʌ'}
    rows = []
    for label in labels:
        index, label_name = label.split("_")
        index = int(index)

        for audio_path in (qolt_path / label).glob("*.wav"):
            grid_path = audio_path.with_suffix(".TextGrid")
            audio_len = librosa.get_duration(filename=audio_path, sr=16000)
            grid = praatio.textgrid.openTextgrid(grid_path, includeEmptyIntervals=True)
            for entry in grid.getTier("phones").entries:
                rows.append({
                    "audio": audio_path,
                    "label": index,
                    "label_name": label_name,
                    "min": entry.start,
                    "max": entry.end,
                    "phone": vocab[entry.label],
                })
            if entry.end < audio_len:
                rows.append({
                    "audio": audio_path,
                    "label": index,
                    "label_name": label_name,
                    "min": entry.end,
                    "max": audio_len,
                    "phone": "(...)",
                })

    return pd.DataFrame(rows)


def _prepare_l2arctic(l2arctic_path: Path):
    rows = []
    def _remove_digits(p):
        return "".join([c for c in p if c not in "0123456789"])

    vocab = {'': '(...)', 'AA': 'ɑ', 'AE': 'æ', 'AH': 'ʌ', 'AO': 'ɔ', 'AW': 'aʊ', 'AY': 'aɪ', 'B': 'b', 'CH': 'tʃ', 'D': 'd', 'DH': 'ð', 'EH': 'ɛ', 'ER': 'ɜː', 'EY': 'eɪ', 'F': 'f', 'G': 'ɡ', 'HH': 'h', 'IH': 'ɪ', 'IY': 'i', 'JH': 'dʒ', 'K': 'k', 'L': 'lʲ', 'M': 'm', 'N': 'n', 'NG': 'ŋ', 'OW': 'əʊ', 'OY': 'ɔɪ', 'P': 'p', 'R': 'r', 'S': 's', 'SH': 'ʃ', 'T': 't', 'TH': 'θ', 'UH': 'ʊ', 'UW': 'u', 'V': 'v', 'W': 'w', 'Y': 'j', 'Z': 'z', 'ZH': 'ʒ', 'sil': '(...)', 'sp': '(...)', 'spn': '(...)'}

    for grid_path in l2arctic_path.glob("*/annotation/*.TextGrid"):
        audio_path = str(grid_path).replace("/annotation/", "/wav/").replace(".TextGrid", ".wav")
        grid = praatio.textgrid.openTextgrid(grid_path, includeEmptyIntervals=True)
        speaker = grid_path.parent.parent.name
        if speaker in ("NJS", "TLV", "TNI", "ZHAA"):
            split = "test"
        elif speaker in ("TXHC", "YKWK"):
            split = "dev"
        else:
            split = "train"
        for entry in grid.getTier("phones").entries:
            emin = entry.xmin if hasattr(entry, "xmin") else entry.start
            emax = entry.xmax if hasattr(entry, "xmax") else entry.end
            phone = entry.text if hasattr(entry, "text") else entry.label
            rows.append({
                "audio": audio_path,
                "speaker": speaker,
                "split": split,
                "min": emin,
                "max": emax,
                "phone": vocab[_remove_digits(phone)],
            })
    return pd.DataFrame(rows)


class PhoneRecognitionDataset(torch.utils.data.Dataset):
    """
    * What should be saved in CSV file?
    We read the "audio" column and "label" column via pandas.read_csv.
    "audio" column should contain the absolute path to the audio, and "label" should be integer: 0, 1, 2.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        sample_rate: int = 16000
    ):
        self.df = df
        self.audios = df.audio.unique()
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.audios)

    def __getitem__(self, i):
        x, _ = librosa.load(self.audios[i], sr=self.sample_rate, mono=True)
        y = self.df[self.df["audio"] == self.audios[i]]

        return x, y[["phone", "min", "max"]]


if __name__ == "__main__":
    args = _get_args()
    _prepare = {"commonphone": _prepare_commonphone, "ssnce": _prepare_ssnce, "torgo": _prepare_torgo, "qolt": _prepare_qolt, "l2arctic": _prepare_l2arctic, "uaspeech": _prepare_uaspeech}[args.dataset_type]
    df = _prepare(args.dataset_path)
    csv_path = args.output_path / f"{args.dataset_type}.csv.gz"
    df.to_csv(csv_path, index=False, compression="gzip")

In [ ]:
%%writefile /content/dysarthria-gop/model.py
import torch
from transformers import AutoModelForPreTraining


class BaseRecognizer(torch.nn.Module):
    def __init__(self, model, feature_size, vocab_size):
        super().__init__()
        self.head = torch.nn.Linear(feature_size, vocab_size)

    def _get_features(self, input_values, attention_mask):
        raise NotImplementedError

    def get_feat_length(self, input_length):
        raise NotImplementedError

    def forward(self, inputs):
        features = self._get_features(inputs)
        logits = self.head(features)
        return logits, features


class Wav2Vec2Recognizer(BaseRecognizer):
    def __init__(self, model, vocab_size):
        feature_size = {
            "facebook/wav2vec2-base": 256,
            "facebook/wav2vec2-xls-r-300m": 1024,
        }[model]
        super().__init__(model, feature_size, vocab_size)
        self.net = AutoModelForPreTraining.from_pretrained(model)
        self.get_feat_length = self.net._get_feat_extract_output_lengths

    def freeze_conv_features(self):
        self.net.freeze_feature_encoder()

    def _get_features(self, inputs):
        # argon_gpu full-transformer patch: self.net(inputs)[0] returns projected_states
        # (768-d), which mismatches the 1024-d head built for this model. Use
        # self.net.wav2vec2(inputs)[0] instead, which is the last_hidden_state (1024-d)
        # of the underlying wav2vec2 transformer.
        outputs = self.net.wav2vec2(inputs)
        return outputs[0]


class Wav2Vec2ConvRecognizer(BaseRecognizer):
    def __init__(self, model, vocab_size):
        feature_size = {
            "facebook/wav2vec2-base": 512,
            "facebook/wav2vec2-xls-r-300m": 512,
        }[model]
        super().__init__(model, feature_size, vocab_size)
        net = AutoModelForPreTraining.from_pretrained(model)
        self.extractor = net.wav2vec2.feature_extractor
        self.projector = net.wav2vec2.feature_projection
        self.get_feat_length = net._get_feat_extract_output_lengths

    def _get_features(self, inputs):
        feats = self.extractor(inputs)
        feats = feats.transpose(1, 2)
        _, feats = self.projector(feats)
        return feats

    def freeze_conv_features(self):
        for module in (self.extractor, self.projector):
            for param in module.parameters():
                param.requires_grad = False

In [ ]:
%%writefile /content/dysarthria-gop/loss.py
import torch
import torch.nn.functional as F


def _get_losses_lengths(logits, labels, log: bool = True):
    # F.nll_loss expects LOG-probabilities as input. Passing plain softmax
    # probabilities (log=False) silently reproduces the old (incorrect) loss;
    # log=True (default) uses log_softmax, which is the mathematically correct fix.
    if log:
        probs = F.log_softmax(logits.transpose(1, 2), dim=1)
    else:
        probs = F.softmax(logits.transpose(1, 2), dim=1)
    losses = F.nll_loss(probs, labels, ignore_index=-100, reduction="none")
    lengths = (labels >= 0).sum(1)
    return losses, lengths


def samplewise_average_loss(logits, labels, log: bool = True):
    losses, lengths = _get_losses_lengths(logits, labels, log=log)
    weights = (1/lengths)[:, None].expand(labels.shape) / len(labels)
    return (losses * weights).sum()


def phonewise_average_loss(logits, labels, log: bool = True):
    losses, lengths = _get_losses_lengths(logits, labels, log=log)
    weights = 1 / lengths.sum()
    return (losses * weights).sum()


def ctc_like_loss(logits, labels, log: bool = True):
    losses, lengths = _get_losses_lengths(logits, labels, log=log)
    weights = []
    for label, length in zip(labels, lengths):
        label = label[:length]
        _, indices, counts = torch.unique_consecutive(
            label, return_inverse=True, return_counts=True)
        weight = torch.index_select(1 / counts, 0, indices) / counts.shape[0]
        weight = torch.cat((weight, torch.zeros(labels.shape[1] - length, device=weight.device)))
        weights.append(weight)
    weights = torch.stack(weights) / len(labels)
    return (losses * weights).sum()


def get_loss(name):
    return {
        "samplewise_average": samplewise_average_loss,
        "phonewise_average": phonewise_average_loss,
        "ctc_like": ctc_like_loss,
    }[name]

In [ ]:
%%writefile /content/dysarthria-gop/train_phone_recognizer.py
import argparse
import pickle
from collections import defaultdict
from datetime import datetime
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn import metrics

import torch
from torch.utils.tensorboard import SummaryWriter
from transformers import Wav2Vec2FeatureExtractor

from dataset import PhoneRecognitionDataset, get_vocab, reduce_vocab
from model import Wav2Vec2Recognizer, Wav2Vec2ConvRecognizer
from loss import get_loss


def _str2bool(v):
    # argparse's type=bool is broken for CLI flags: bool("False") == True since any
    # non-empty string is truthy. This helper parses common textual boolean forms instead.
    if isinstance(v, bool):
        return v
    if v.lower() in ("yes", "true", "t", "y", "1"):
        return True
    if v.lower() in ("no", "false", "f", "n", "0"):
        return False
    raise argparse.ArgumentTypeError(f"Boolean value expected, got {v!r}")


def _get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--exp_name", type=str)

    # Training Settings
    parser.add_argument("--gpu", default=None, type=int, help="Default to CPU. Input GPU index (integer) to use GPU.")
    parser.add_argument("--bestkeep_metric", default="accuracy", type=str)
    parser.add_argument("--num_epochs", default=10, type=int)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--loss", type=str, default="ctc_like")

    # Model Settings
    parser.add_argument("--model", default="facebook/wav2vec2-xls-r-300m", type=str)
    parser.add_argument("--use_conv_only", default=True, type=_str2bool)

    # Optimizer Settings
    parser.add_argument("--optim", default="AdamW", type=str)
    parser.add_argument("--learning_rate", type=float, default=5e-5)

    # Dataset settings
    parser.add_argument("--commonphone_csv", required=True, type=Path)
    parser.add_argument("--reduce_vocab", default=False, type=_str2bool)

    return parser.parse_args()


def _prepare_model(model_name, vocab_size, use_conv_only):
    ModelClass = {False: Wav2Vec2Recognizer, True: Wav2Vec2ConvRecognizer}[use_conv_only]
    model = ModelClass(model_name, vocab_size)
    model.freeze_conv_features()
    return model


def _get_logger(tb_path):
    writer = SummaryWriter(log_dir=tb_path)
    step_acc = defaultdict(int)
    def _log(name, value):
        writer.add_scalar(name, value, step_acc[name])
        step_acc[name] += 1
    return _log


def _get_collator(model, vocab_to_index, _get_feat_extract_output_lengths):
    processor = Wav2Vec2FeatureExtractor.from_pretrained(model)
    def _collate(batch):
        # Record each sample's true (unpadded) waveform length before the processor
        # pads audios with zeros, so we can later derive the true feature length.
        raw_lengths = [len(b[0]) for b in batch]
        audios = [b[0] for b in batch]
        audios = processor(raw_speech=audios, sampling_rate=16000, padding=True)
        audios = torch.FloatTensor(audios["input_values"])

        batch_size, max_length = audios.shape
        max_feature_length = _get_feat_extract_output_lengths(max_length).item()

        labels = np.ones((batch_size, max_feature_length), dtype=np.int32) * -100
        for i, (_, _df) in enumerate(batch):
            # The old code compared padded audio samples to the -100 label sentinel, but
            # padding uses 0.0 and this conflates waveform samples with feature frames.
            # Instead, derive the true feature length from the true (unpadded) raw waveform
            # length, clamped to max_feature_length so padded frames stay -100 (ignored).
            feature_length = min(_get_feat_extract_output_lengths(raw_lengths[i]).item(), max_feature_length)
            labels[i, 0:feature_length] = vocab_to_index["(...)"]

            for _, row in _df.iterrows():
                index = vocab_to_index[row["phone"]]
                start_loc = _get_feat_extract_output_lengths(int(row["min"] * 16000)).item()
                end_loc = min(_get_feat_extract_output_lengths(int(row["max"] * 16000)).item(), feature_length)

                if start_loc < end_loc:
                    labels[i, start_loc:end_loc] = index

        labels = torch.LongTensor(labels)

        return audios, labels

    return _collate


def _prepare_data(df, batch_size, collator):
    train_ds = torch.utils.data.DataLoader(
        PhoneRecognitionDataset(df[df.split == "train"]),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True,
        num_workers=2,
        collate_fn=collator,
    )
    valid_ds = torch.utils.data.DataLoader(
        PhoneRecognitionDataset(df[df.split == "dev"]),
        batch_size=batch_size,
        shuffle=False,
        pin_memory=True,
        num_workers=2,
        collate_fn=collator,
    )
    test_ds = torch.utils.data.DataLoader(
        PhoneRecognitionDataset(df[df.split == "test"]),
        batch_size=batch_size,
        shuffle=False,
        pin_memory=True,
        num_workers=2,
        collate_fn=collator,
    )
    return train_ds, valid_ds, test_ds


def _train(model, device, optim, loss_fn, dataloader, logger):
    model.train()

    correct_acc = 0
    wrong_acc = 0
    loss_acc = 0.0

    for x, y in tqdm(dataloader):
        x = x.to(device)
        y = y.to(device)

        optim.zero_grad()
        logits, _ = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optim.step()

        logger("train/loss", loss.item())
        loss_acc += loss.item()

        correct_acc += ((logits.detach().argmax(-1) == y) * (y >= 0)).sum()
        wrong_acc += ((logits.detach().argmax(-1) != y) * (y >= 0)).sum()

    logger("train/avg_loss", loss_acc / len(dataloader))
    logger("train/accuracy", correct_acc / (correct_acc + wrong_acc))


def _eval(model, device, dataloader, logger, metric_funcs, mode):
    model.eval()

    preds_acc = []
    labels_acc = []
    for x, y in tqdm(dataloader):
        logits, _ = model(x.to(device))
        preds_acc.append(logits.detach().softmax(-1).cpu().numpy())
        labels_acc.append(y.numpy())

    eval_results = {}
    for name, func in metric_funcs.items():
        eval_results[name] = func(labels_acc, preds_acc)
        logger(f"{mode}/{name}", eval_results[name])

    return {"preds": preds_acc, "labels": labels_acc, "metrics": eval_results}


def _discretize_metric(metric):
    def _metric(y_true, y_pred):
        y_true = np.concatenate([l.flatten() for l in y_true])
        y_pred = np.concatenate([p.argmax(-1).flatten() for p in y_pred])
        mask = y_true >= 0
        return metric(y_true[mask], y_pred[mask])
    return _metric


_metrics = {
    "accuracy": _discretize_metric(metrics.accuracy_score),
}


if __name__ == "__main__":
    args = _get_args()
    print(args)

    exp_dir = Path("exp") / f"{args.exp_name}_{datetime.today().isoformat()}"
    epoch_dir = exp_dir / "epochs"
    epoch_dir.mkdir(exist_ok=False, parents=True)

    logger = _get_logger(exp_dir / "logs")
    device = torch.device("cpu" if args.gpu is None else args.gpu)

    df = pd.read_csv(args.commonphone_csv, compression="gzip")
    if args.reduce_vocab:
        df = reduce_vocab(df)
    index_to_vocab, vocab_to_index = get_vocab(df)

    model = _prepare_model(args.model, len(index_to_vocab), args.use_conv_only).to(device)

    collator = _get_collator(args.model, vocab_to_index, model.get_feat_length)
    train_dataloader, valid_dataloader, test_dataloader = _prepare_data(df, args.batch_size, collator)

    optim = getattr(torch.optim, args.optim)(model.parameters(), lr=args.learning_rate)
    loss_fn = get_loss(args.loss)

    torch.save(model.state_dict(), exp_dir / "best.pt")
    best_epoch, best_metric = None, None
    for epoch in range(args.num_epochs):
        _train(model, device, optim, loss_fn, train_dataloader, logger)
        _eval_results = _eval(model, device, valid_dataloader, logger, _metrics, "valid")

        print(f"Epoch {epoch}")
        print(_eval_results["metrics"])

        epochwise_dir = epoch_dir / f"{epoch:04d}"
        epochwise_dir.mkdir(exist_ok=False, parents=True)
        pickle.dump(_eval_results, open(epochwise_dir / "eval_results.pkl", "wb"))

        if best_epoch is None or best_metric < _eval_results["metrics"][args.bestkeep_metric]:
            best_epoch = epoch
            best_metric = _eval_results["metrics"][args.bestkeep_metric]
            torch.save(model.state_dict(), exp_dir / "best.pt")

    model.load_state_dict(torch.load(exp_dir / "best.pt"))
    _test_results = _eval(model, device, test_dataloader, logger, _metrics, "test")
    pickle.dump(_test_results, open(exp_dir / "test_results.pkl", "wb"))
    pickle.dump(index_to_vocab, open(exp_dir / "index_to_vocab.pkl", "wb"))
    pickle.dump(args, open(exp_dir / "arguments.pkl", "wb"))

    print("Training Finished!")
    print(_test_results)

In [ ]:
%%writefile /content/dysarthria-gop/port_checkpoint.py
"""Port a GPU/Linux training checkpoint dir so it loads on CPU/Windows.

Two independent portability shims (see PROJECT_CONTEXT.md sec 8):

  (a) `arguments.pkl` may contain `pathlib.PosixPath` objects, which raise
      `NotImplementedError: cannot instantiate PosixPath` when unpickled on a
      non-POSIX platform. Fixed by unpickling with a custom `Unpickler` that
      remaps `pathlib.PosixPath` -> `pathlib.PurePosixPath` (a pure,
      platform-independent path type that can always be instantiated), then
      coercing any path-typed attributes on the resulting object to plain
      `str` and re-pickling in place.

  (b) `best.pt` may hold CUDA tensors, which raise `RuntimeError: Attempting
      to deserialize object on a CUDA device` when loaded without a GPU.
      Fixed with `torch.save(torch.load(path, map_location="cpu"), path)`.

torch is imported LAZILY (only inside the function that needs it) so this
module can be imported / the arguments.pkl shim can run in environments
without torch installed. No model weights are downloaded.

CLI:
    python port_checkpoint.py --model_dir path/to/exp/run_dir
"""

import argparse
import pathlib
import pickle
from pathlib import Path


class _PosixToPureUnpickler(pickle.Unpickler):
    """Unpickler that maps pathlib.PosixPath -> pathlib.PurePosixPath.

    PosixPath can't be instantiated on Windows (or vice versa for WindowsPath
    on POSIX); PurePosixPath is a pure path type with no OS dependency, so it
    can always be constructed regardless of the host platform.
    """

    def find_class(self, module, name):
        if module == "pathlib" and name == "PosixPath":
            return pathlib.PurePosixPath
        if module == "pathlib" and name == "WindowsPath":
            return pathlib.PureWindowsPath
        return super().find_class(module, name)


def _coerce_path_attrs_to_str(obj):
    """Walk obj.__dict__ (if any) and stringify any path-like attribute values."""
    d = getattr(obj, "__dict__", None)
    if d is None:
        return obj
    for key, value in list(d.items()):
        if isinstance(value, (pathlib.PurePath,)):
            d[key] = str(value)
    return obj


def port_arguments_pkl(model_dir: Path) -> Path:
    """Re-pickle `model_dir/arguments.pkl` with PosixPath shimmed + path attrs stringified.

    Returns the path that was rewritten.
    """
    args_path = Path(model_dir) / "arguments.pkl"
    with open(args_path, "rb") as f:
        args = _PosixToPureUnpickler(f).load()

    args = _coerce_path_attrs_to_str(args)

    with open(args_path, "wb") as f:
        pickle.dump(args, f)

    return args_path


def port_best_pt(model_dir: Path) -> Path:
    """Reload `model_dir/best.pt` mapped to CPU and re-save it in place.

    Imports torch lazily; raises a clear error if torch is not installed.
    """
    try:
        import torch
    except ImportError as e:
        raise ImportError(
            "port_best_pt requires torch, which is not installed in this "
            "environment. Install torch (CPU build is fine) to port best.pt."
        ) from e

    best_pt_path = Path(model_dir) / "best.pt"
    state = torch.load(best_pt_path, map_location="cpu")
    torch.save(state, best_pt_path)
    return best_pt_path


def port_checkpoint(model_dir: Path) -> dict:
    """Run both shims against a checkpoint directory. Returns paths touched."""
    model_dir = Path(model_dir)
    result = {}

    args_path = model_dir / "arguments.pkl"
    if args_path.exists():
        result["arguments_pkl"] = str(port_arguments_pkl(model_dir))
    else:
        result["arguments_pkl"] = None

    best_pt_path = model_dir / "best.pt"
    if best_pt_path.exists():
        result["best_pt"] = str(port_best_pt(model_dir))
    else:
        result["best_pt"] = None

    return result


def _get_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model_dir", type=Path, required=True,
                         help="Directory containing arguments.pkl and/or best.pt")
    return parser.parse_args()


def main():
    args = _get_args()
    result = port_checkpoint(args.model_dir)

    if result["arguments_pkl"]:
        print(f"Re-pickled: {result['arguments_pkl']}")
    else:
        print(f"No arguments.pkl found in {args.model_dir}; skipped.")

    if result["best_pt"]:
        print(f"Re-saved (CPU): {result['best_pt']}")
    else:
        print(f"No best.pt found in {args.model_dir}; skipped.")


if __name__ == "__main__":
    main()

In [ ]:
%cd /content/dysarthria-gop

## 6. Download CMU ARCTIC (public)

Each speaker release is ~1100 utterances of 16 kHz read speech, with prompts in

`etc/txt.done.data`.

In [ ]:
import os, subprocess, urllib.request, tarfile
ARCTIC_DIR = '/content/arctic'
os.makedirs(ARCTIC_DIR, exist_ok=True)
for spk in SPEAKERS:
    dest = f'{ARCTIC_DIR}/cmu_us_{spk}_arctic'
    if os.path.isdir(dest + '/wav'):
        print('already have', spk); continue
    url = f'http://festvox.org/cmu_arctic/packed/cmu_us_{spk}_arctic-0.95-release.tar.bz2'
    tb = f'{ARCTIC_DIR}/{spk}.tar.bz2'
    print('downloading', url)
    urllib.request.urlretrieve(url, tb)
    with tarfile.open(tb, 'r:bz2') as t:
        t.extractall(ARCTIC_DIR)
    os.remove(tb)
    print('extracted', spk)
print('done:', SPEAKERS)

## 7. Build the MFA corpus (wav + .lab transcript per utterance)

MFA reads a folder of `<speaker>/<uttid>.wav` + `<uttid>.lab` (the transcript).

In [ ]:
import os, re, shutil
CORPUS = '/content/mfa_corpus'
shutil.rmtree(CORPUS, ignore_errors=True)
prompt_re = re.compile(r'\(\s*(\S+)\s+"(.*)"\s*\)')
n = 0
for spk in SPEAKERS:
    rel = f'/content/arctic/cmu_us_{spk}_arctic'
    spk_out = f'{CORPUS}/{spk}'
    os.makedirs(spk_out, exist_ok=True)
    with open(f'{rel}/etc/txt.done.data') as f:
        for line in f:
            m = prompt_re.search(line)
            if not m:
                continue
            uttid, text = m.group(1), m.group(2)
            wav = f'{rel}/wav/{uttid}.wav'
            if not os.path.exists(wav):
                continue
            shutil.copy(wav, f'{spk_out}/{uttid}.wav')
            with open(f'{spk_out}/{uttid}.lab', 'w') as g:
                g.write(text)
            n += 1
print('prepared', n, 'utterances across', len(SPEAKERS), 'speakers')

## 8. Run MFA alignment (english_mfa IPA acoustic + dictionary)

This produces one TextGrid per utterance with an IPA `phones` tier that matches the

`reduce_vocab` reducer in `dataset.py`. First run downloads the models (~1 min);

alignment of ~2k utts takes a few minutes.

In [ ]:
!mfa model download acoustic english_mfa 2>&1 | tail -2
!mfa model download dictionary english_us_mfa 2>&1 | tail -2
ALIGNED = '/content/mfa_aligned'
!rm -rf {ALIGNED}
!mfa align --clean -j 4 /content/mfa_corpus english_us_mfa english_mfa {ALIGNED} 2>&1 | tail -15
import glob
print('TextGrids produced:', len(glob.glob(ALIGNED + '/**/*.TextGrid', recursive=True)))

## 9. Build the training CSV from the aligned TextGrids

Rows: `audio, phone, min, max, split`. Silence/OOV tokens map to `(...)`. Split is

deterministic by utterance id (~80/10/10) — the 'strong', split-by-utterance style.

In [ ]:
import glob, os, hashlib
import pandas as pd
from praatio import textgrid as tgio

SIL = {'', 'sil', 'sp', 'spn', 'spn', '<eps>', 'noise'}
def split_of(uttid):
    h = int(hashlib.md5(uttid.encode()).hexdigest(), 16) % 100
    return 'train' if h < 80 else ('dev' if h < 90 else 'test')

rows = []
for tg_path in glob.glob(ALIGNED + '/**/*.TextGrid', recursive=True):
    uttid = os.path.splitext(os.path.basename(tg_path))[0]
    spk = os.path.basename(os.path.dirname(tg_path))
    wav = f'/content/arctic/cmu_us_{spk}_arctic/wav/{uttid}.wav'
    if not os.path.exists(wav):
        wav = f'/content/mfa_corpus/{spk}/{uttid}.wav'
    tg = tgio.openTextgrid(tg_path, includeEmptyIntervals=True)
    tier_name = next((n for n in tg.tierNames if 'phone' in n.lower()), None)
    if tier_name is None:
        continue
    for entry in tg.getTier(tier_name).entries:
        phone = entry.label.strip()
        phone = '(...)' if phone in SIL else phone
        rows.append({'audio': wav, 'phone': phone, 'min': entry.start,
                     'max': entry.end, 'split': split_of(uttid)})
df = pd.DataFrame(rows)
CSV = '/content/arctic_mfa.csv.gz'
df.to_csv(CSV, index=False, compression='gzip')
print('rows:', len(df), '| unique phones:', df.phone.nunique())
print(df.split.value_counts().to_dict())

### 9b. Sanity-check phones against the reducer

`reduce_vocab` maps english_mfa IPA to the 39-phone set. Report any phone it can't map.

In [ ]:
import sys; sys.path.insert(0, '/content/dysarthria-gop')
from dataset import reduce_vocab
import pandas as pd
# reduce_vocab raises KeyError on any phone missing from its reducer; test each phone.
unmapped = []
for p in df.phone.unique():
    try:
        reduce_vocab(pd.DataFrame({'phone': [p]}))
    except KeyError:
        unmapped.append(p)
if unmapped:
    print('Unmapped phones -> remapping to (...):', unmapped)
    df.loc[df.phone.isin(unmapped), 'phone'] = '(...)'
    df.to_csv(CSV, index=False, compression='gzip')
    print('remapped', len(unmapped), 'phone type(s) and rewrote', CSV)
else:
    print('OK: all', df.phone.nunique(), 'phones map cleanly into the reduced set.')

## 10. Train the conv-only recognizer (GPU)

Only the linear phone head trains (conv extractor + projection are frozen), so this

is light. Watch `valid/accuracy` climb.

In [ ]:
!cd /content/dysarthria-gop && python train_phone_recognizer.py \
    --exp_name {EXP_NAME} \
    --num_epochs {NUM_EPOCHS} \
    --learning_rate {LR} \
    --loss phonewise_average \
    --gpu 0 \
    --model facebook/wav2vec2-xls-r-300m \
    --use_conv_only True \
    --batch_size {BATCH_SIZE} \
    --commonphone_csv /content/arctic_mfa.csv.gz \
    --reduce_vocab True

## 11. Export the checkpoint (ported for local CPU/Windows)

Copies `best.pt`, `index_to_vocab.pkl`, `arguments.pkl` into `models/`, runs

`port_checkpoint.py` (PosixPath -> str, CUDA -> CPU), then zips for download.

In [ ]:
import glob, os, shutil
runs = sorted(glob.glob('/content/dysarthria-gop/exp/' + EXP_NAME + '_*'))
assert runs, 'no exp run found — did training finish?'
run = runs[-1]
OUT_MODEL = '/content/dysarthria-gop/models/' + EXP_NAME
os.makedirs(OUT_MODEL, exist_ok=True)
for f in ('best.pt', 'index_to_vocab.pkl', 'arguments.pkl'):
    shutil.copy(os.path.join(run, f), os.path.join(OUT_MODEL, f))
print('registered from', run)
!cd /content/dysarthria-gop && python port_checkpoint.py --model_dir models/{EXP_NAME}
shutil.copy('/content/arctic_mfa.csv.gz', os.path.join(OUT_MODEL, 'arctic_mfa.csv.gz'))  # prior source for gop.py
zip_path = shutil.make_archive('/content/' + EXP_NAME + '_model', 'zip', OUT_MODEL)
print('zipped ->', zip_path)

### 11b. Download the checkpoint (or save to Google Drive)

In [ ]:
from google.colab import files
files.download('/content/' + EXP_NAME + '_model.zip')

# --- or copy to Google Drive instead (uncomment) ---
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copy('/content/' + EXP_NAME + '_model.zip', '/content/drive/MyDrive/')

## 12. Use it locally

On your data-bearing machine, drop the unzipped folder in as `models/<name>/` and run:

```bash

python gop.py --dataset_csv test_dataset.csv.gz \

    --commonphone_csv arctic_mfa.csv.gz \

    --model_path models/arctic_conv_colab --temperature <refit>

python validate_gop.py --outputs_pkl test_dataset.csv_outputs.pkl --crosswalk crosswalk_anon_to_speaker.csv

python mild_band_auc.py --outputs_pkl test_dataset.csv_outputs.pkl --crosswalk crosswalk_anon_to_speaker.csv

```

Re-fit `--temperature` for this checkpoint with `temperature_scaling.py` (the old

7.8375 was fit on a different model). The bundled `arctic_mfa.csv.gz` supplies the

phone prior for the Norm*/DNN scorers.